<a href="https://colab.research.google.com/github/LuGorr/NLP-Assignments/blob/main/A1/Assignment_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Group members

|  Name   |  Surname   |     Email                            |    Student ID                                             |
| :-----: | :--------: | :----------------------------------: | :-----------------------------------------------------: |
| Ludovico  | Gorrieri   | `ludovico.gorrieri@studio.unibo.it`   |  TBD |
| Alessandro  | Capialbi | `alessandro.capialbi@studio.unibo.it`  | 0001191564 |
| Faezeh  | Sarlakifar | `faezeh.sarlakifar@studio.unibo.it`  | 0001164608 |

## Task 1 & 2

### Download the dataset

In [4]:
# !wget https://github.com/nlp-unibo/nlp-course-material/tree/main/2025-2026/Assignment%201/data

In [5]:
!git clone https://github.com/nlp-unibo/nlp-course-material.git
%cd "nlp-course-material/2025-2026/Assignment 1"

Cloning into 'nlp-course-material'...
remote: Enumerating objects: 391, done.
remote: Counting objects: 100% (391/391), done.
remote: Compressing objects: 100% (288/288), done.
remote: Total 391 (delta 174), reused 294 (delta 90), pack-reused 0 (from 0)
Receiving objects: 100% (391/391), 8.56 MiB | 17.64 MiB/s, done.
Resolving deltas: 100% (174/174), done.
/content/nlp-course-material/2025-2026/Assignment 1/nlp-course-material/2025-2026/Assignment 1


### **Tweet Preprocessing and Label Aggregation Script**

This script prepares the dataset of tweets for NLP tasks.
It handles text cleaning, tokenization, lemmatization, and label aggregation for supervised learning.
Below is a detailed explanation of each section.

#### 1. Importing Required Libraries:

    a) pandas / numpy → for data manipulation.
    b) re → regular expressions for text cleaning.
    c) nltk → for tokenization, POS tagging, and lemmatization.
    d) Counter → to count occurrences of labels and select the majority vote.

#### 2. Preparing NLTK Resources:

    This block ensures that all required NLTK corpora and models are available locally.
    If a resource is missing, it is automatically downloaded.

#### 3. Initializing Tools:

    WhitespaceTokenizer → splits text based on spaces (useful after cleaning).
    WordNetLemmatizer → reduces words to their base or dictionary form using WordNet.

#### 4. Helper Function: get_wordnet_key(pos_tag):
    This function maps Penn Treebank POS tags (e.g., NN, VB, JJ) to WordNet’s format (noun, verb, adjective, adverb).
    This step is essential because WordNetLemmatizer requires the part of speech to perform accurate lemmatization.

#### 5. Lemmatization Function: lem_text(row):
    Reduces words to their dictionary form (lemma) using POS information.
    
    This function:

    1) Tokenizes the tweet into words.
    2) Assigns POS tags using NLTK’s pos_tag.
    3) Lemmatizes each word according to its part of speech.
    4) Returns the lemmatized tweet as a single string.

#### 6. Cleaning Function: cleaner(row):

    Purpose: Remove noise and standardize text before analysis.

    Steps:

    1) lower(): Converts all text to lowercase
    2) Remove URLs: Regex https?:\/\/.\S+ removes URLs and links
    3) Remove mentions & hashtags:	Regex [@#].\S+ removes @user and #topic
    4) Remove emojis/symbols: Unicode ranges cover emoticons, flags, pictographs
    5) Remove non-alphanumeric:	Keeps only letters, digits, and spaces
    6) Normalize whitespace:	Collapses multiple spaces into one

#### 7. Label Aggregation:

    This part aggregates multiple label votes for a tweet into a single numeric label.

    How it works:

    1) For each row, it collects all values in labels_task2 except "UNKNOWN".
    2) Uses Counter to find the most common label (majority vote).
    3) Maps that label to a numerical ID using the mapping dictionary.

#### Summary

    This script prepares tweets by performing:

    1) cleaner():	Remove unwanted characters and normalize text
    2) lem_text(): Lemmatize words for consistent representation
    3) aggregator(): Convert multiple annotations into a single label

    Together, these functions create a clean, normalized, and labeled dataset,
    ideal for tasks like text classication that we will perform.


In [6]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 82.6 MB/s eta 0:00:00


In [7]:
# Imports
import os
import json
import re
import numpy as np
import pandas as pd
from collections import Counter

# NLTK setup
import nltk
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize, sent_tokenize, WhitespaceTokenizer

# TensorFlow & Keras
import tensorflow as tf
from tensorflow.keras import layers, models

# Gensim / GloVe
import gensim
from gensim.models import KeyedVectors

# Download GloVe (one time)
# !wget http://nlp.stanford.edu/data/glove.twitter.27B.zip
# !unzip -q glove.twitter.27B.zip


In [8]:
# Prepare NLTK resources
resources = [
    ('corpora/omw-1.4', 'omw-1.4'),
    ('corpora/wordnet', 'wordnet'),
    ('taggers/averaged_perceptron_tagger', 'averaged_perceptron_tagger'),
    ('taggers/averaged_perceptron_tagger_eng', 'averaged_perceptron_tagger_eng'),
    ('tokenizers/punkt_tab', 'punkt_tab'),
    ('tokenizers/punkt', 'punkt')
]

# Ensure that all the required NLTK resources are downloaded.
# If a resource is missing, it is downloaded quietly (without console output).
for resource_path, download_name in resources:
    try:
        nltk.data.find(resource_path)
    except LookupError:
        nltk.download(download_name, quiet=True)

# Initialize tokenizer and lemmatizer
tokenizer = WhitespaceTokenizer()
lemmatizer = WordNetLemmatizer()

def get_wordnet_key(pos_tag):
    """
    Map POS (Part-of-Speech) tags from the Penn Treebank format
    to the corresponding WordNet POS tags.

    Args:
        pos_tag (str): The POS tag from NLTK's `pos_tag` function.

    Returns:
        str: Corresponding WordNet POS tag (ADJ, VERB, NOUN, ADV).
             Defaults to 'n' (noun) if no match is found.
    """
    if pos_tag.startswith('J'):
        return wordnet.ADJ
    elif pos_tag.startswith('V'):
        return wordnet.VERB
    elif pos_tag.startswith('N'):
        return wordnet.NOUN
    elif pos_tag.startswith('R'):
        return wordnet.ADV
    else:
        return 'n'

def lem_text(row):
    """
    Lemmatize a tweet using NLTK's WordNet lemmatizer.

    Steps:
        1. Tokenize the tweet by whitespace.
        2. Tag each token with its POS.
        3. Lemmatize each token using its WordNet POS tag.
        4. Join the lemmatized tokens back into a single string.

    Args:
        row (pd.Series): A row from a pandas DataFrame containing a 'tweet' field.

    Returns:
        str: The lemmatized version of the tweet.
    """
    tokens = tokenizer.tokenize(row.tweet)
    tagged = pos_tag(tokens)
    words = [lemmatizer.lemmatize(word, get_wordnet_key(tag))
             for word, tag in tagged]
    return " ".join(words)

def cleaner(row):
    """
    Normalize and clean a tweet by removing unwanted elements.

    Steps:
        - Convert to lowercase.
        - Remove URLs, mentions, and hashtags.
        - Remove emojis and special Unicode characters.
        - Remove non-alphanumeric characters.
        - Remove excessive whitespace.

    Args:
        row (pd.Series): A row from a pandas DataFrame containing a 'tweet' field.

    Returns:
        str: The cleaned tweet text.
    """
    text = row.tweet
    text = text.lower()
    text = re.sub(r'https?:\/\/.\S+', '', text)        # Remove URLs
    text = re.sub(r'[@#].\S+', '', text)               # Remove mentions and hashtags
    text = re.sub(
        "["                                             # Remove emojis and symbols
        u"\U0001F600-\U0001F64F"  # Emoticons
        u"\U0001F300-\U0001F5FF"  # Symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # Transport & map symbols
        u"\U0001F1E0-\U0001F1FF"  # Flags
        "]+", '', text
    )
    text = re.sub(r'[^a-z^0-9^\s]*', '', text)         # Remove non-alphanumeric chars
    text = ' '.join(text.split())                      # Remove extra spaces
    return text

# Define a mapping for label aggregation
mapping = {
    '-': 0,
    'DIRECT': 1,
    'JUDGEMENTAL': 2,
    'REPORTED': 3
}

# Aggregate multiple labels into a single class value based on majority voting.
# If there are multiple annotations for a tweet, the most common label (excluding "UNKNOWN")
# is selected and mapped to its numeric representation using the 'mapping' dictionary.
aggregator = lambda row: \
    mapping[Counter([vote for vote in row.labels_task2 if vote != "UNKNOWN"]).most_common(1)[0][0]]


### Clean, split and lemmatize the dataset.

In [9]:
# Load the files
with open("data/training.json", "r") as tr, \
     open("data/validation.json", "r") as te, \
     open("data/test.json", "r") as va:
    train_json = json.load(tr)
    val_json = json.load(te)
    test_json = json.load(va)

# Create the dataframes (setting the index to id_EXIST)
dts = {
    "train": pd.DataFrame.from_dict(train_json, orient="index").set_index("id_EXIST"),
    "test": pd.DataFrame.from_dict(test_json, orient="index").set_index("id_EXIST"),
    "val": pd.DataFrame.from_dict(val_json, orient="index").set_index("id_EXIST")
}

# Unnecessary columns
drop_cols = ["number_annotators", "annotators", "gender_annotators",
    "age_annotators", "labels_task1", "labels_task3", "split"]

# Clean and lemmatize the data
for name, df in dts.items():
    df = df[df.lang == "en"] # Drop spanish.

    df = df.drop(columns=drop_cols) # Drop unnecessary cols.

    df["labels"] = df.apply(aggregator, axis=1) # Aggregate the labels (maj. voting).
    df = df.drop(columns="labels_task2")

    for func in [cleaner, lem_text]:
        df["tweet"] = df.apply(func, axis=1) # Clean the tweets.

    dts[name] = df

train, test, val = dts.values()

## Task 3: Text Encoding

# Text Encoding for Neural Sexism Classifier

 The goal is to produce a clear, reproducible pipeline that: builds a vocabulary, initializes embeddings from GloVe where available, creates embeddings for train-set-only tokens, and assigns a special `<UNK>` embedding for new tokens encountered in validation/test sets.

---

## 1. Goal and constraints

* Embed words using pre-trained **GloVe** vectors (we chose `glove.twitter.27B.100d.txt` in the code).
* **All tokens present in the training set must be included in the vocabulary** and must have an embedding (from GloVe if available, or a custom embedding if not).
* Tokens that appear only in validation/test but not in training must be mapped to a special token (for example `<UNK>`), which has a **static** embedding (the same embedding for every OOV token in val/test).
* We may choose how to build the `static/<UNK>` embedding (random, neighbourhood average, or other strategy).

---

## 2. Vocabulary design

1. **Train-only vocabulary**

   * Vocabulary = all distinct tokens from the training set.
   * Guarantees every train token has an entry and an embedding.
   * Any token in val/test not in this vocabulary becomes `<UNK>`.

> We implemented a `TextVectorization` layer to build the vocabulary from the training data.

---

## 3. High-level pipeline

1. **Prepare datasets**

   * Extract `texts` and `labels` for `train`, `val`, `test` and create `tf.data.Dataset` objects batched with `batch(64)`.
   * Example from the code: `train_ds = tf.data.Dataset.from_tensor_slices((texts, labels)).batch(64)`.

2. **Build a TextVectorization layer**

   * `vectorizer = layers.TextVectorization(max_tokens=20000, output_sequence_length=100)`.
   * Fit it on the training texts: `vectorizer.adapt(train_ds.map(lambda x, y: x))`.
   * This produces a vocabulary `vocab = vectorizer.get_vocabulary()`.

3. **Load GloVe**

   * Use Gensim's `KeyedVectors.load_word2vec_format(glove_file, binary=False, no_header=True)` to load `glove.twitter.27B.100d.txt` into memory.
   * `embedding_dim = twitter_glove.vector_size` (100 in the example).

4. **Construct embedding matrix**

   * Initialize `embedding_matrix = np.zeros((len(vocab), embedding_dim))`.
   * For each word in `vocab`:

     * If `word` exists in `twitter_glove`, set `embedding_matrix[i] = twitter_glove[word]`.
     * Else (train-only OOV): set `embedding_matrix[i] = some_custom_embedding` (your code uses `np.random.normal(scale=0.6, size=(embedding_dim,))`).

5. **Create a special `<UNK>` token for val/test OOVs**

   * Ensure the vectorizer (or token-to-index mapping) reserves an index for a special token (e.g., `'<UNK>'`) or use the 0 index for padding and a fixed index (e.g., `1`) for `<UNK>`.
   * In our mapping, tokens in val/test not present in the vocabulary must be converted to the index of `<UNK>` before being fed to the model.
   * Give `<UNK>` a static embedding (recommended: deterministic seed or computed average).

6. **Initialize Keras Embedding layer**

   * `embedding_layer = layers.Embedding(input_dim=len(vocab), output_dim=embedding_dim, embeddings_initializer=tf.keras.initializers.Constant(embedding_matrix), trainable=True)`
   * `trainable=True` allows fine-tuning of pre-trained embeddings and learns OOV embeddings during training.

---

## 4. Details: Handling OOV tokens (train vs. val/test)

* **Token in train set but not in GloVe** (train-only OOV):

  * Must be *added* to the vocabulary and assigned an embedding.
  * Our code initializes these rows with random normal vectors: `np.random.normal(scale=0.6, size=(embedding_dim,))`.
  * Because `embedding_layer` is `trainable=True`, these vectors get adjusted during training.

* **Token not in train set but appears in val/test** (val/test OOV):

  * Must be mapped to `<UNK>` (single special token index) so they don't introduce new indices not covered by `embedding_matrix`.
  * `<UNK>` must have a static embedding which remains constant *until* (optionally) you let the model train it. The task statement asks for a static embedding — we can still allow training, but the wording suggests a fixed initialization.


### Setup

In [10]:
!wget http://nlp.stanford.edu/data/glove.twitter.27B.zip
!unzip -q glove.twitter.27B.zip

--2025-11-03 21:03:01--  http://nlp.stanford.edu/data/glove.twitter.27B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.twitter.27B.zip [following]
--2025-11-03 21:03:01--  https://nlp.stanford.edu/data/glove.twitter.27B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.twitter.27B.zip [following]
--2025-11-03 21:03:02--  https://downloads.cs.stanford.edu/nlp/data/glove.twitter.27B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1520408563 (1.4G) [ap

In [11]:
# Set the Keras backend to TensorFlow
# This ensures that all Keras operations are executed using TensorFlow as the underlying framework.
os.environ["KERAS_BACKEND"] = "tensorflow"


### Build the vocabulary

In [12]:
# Extract tweet texts and corresponding labels from the training set
texts = train["tweet"].values
labels = train["labels"].values

# Create a TensorFlow Dataset for training data
# Each batch contains 64 examples for efficient training
train_ds = tf.data.Dataset.from_tensor_slices((texts, labels)).batch(64)

In [13]:
# Extract validation data (texts and labels)
valid_texts = val["tweet"].values
valid_labels = val["labels"].values

# Create a TensorFlow Dataset for validation data
valid_ds = tf.data.Dataset.from_tensor_slices((valid_texts, valid_labels)).batch(64)

# Extract test data (texts and labels)
test_texts = test["tweet"].values
test_labels = test["labels"].values

# Create a TensorFlow Dataset for test data
test_ds = tf.data.Dataset.from_tensor_slices((test_texts, test_labels)).batch(64)

In [14]:
# Initialize a TextVectorization layer
# - max_tokens: maximum vocabulary size (words beyond this limit are marked as "out of vocabulary")
# - output_sequence_length: maximum sequence length (texts are truncated or padded to this length)
vectorizer = layers.TextVectorization(max_tokens=20000, output_sequence_length=100)

# Adapt the vectorizer to the training data to build the vocabulary
# Only the text input (x) is passed for adaptation
vectorizer.adapt(train_ds.map(lambda x, y: x))

In [15]:
# Apply the same trained vectorizer to all dataset splits (train, validation, and test)
# Each text is transformed into a sequence of integer word indices
train_ds = train_ds.map(lambda x, y: (vectorizer(x), y))
valid_ds = valid_ds.map(lambda x, y: (vectorizer(x), y))
test_ds  = test_ds.map(lambda x, y: (vectorizer(x), y))

In [16]:
# Retrieve the vocabulary from the vectorizer
vocab = vectorizer.get_vocabulary()

# Print the first 10 tokens in the vocabulary for inspection
print(vocab[:10])

['', '[UNK]', np.str_('be'), np.str_('the'), np.str_('a'), np.str_('to'), np.str_('and'), np.str_('of'), np.str_('i'), np.str_('it')]


The vectorizer has its vocabulary frozen after adapt().

When we apply it to the val or test set,

any word not seen during training will automatically be replaced with the OOV token (default **UNK**, index 1).

We do not need to manually preprocess or map OOV tokens, TensorFlow will handle it internally.

### Use GloVe Embedding vectors

#### Convert GloVe format to Word2Vec format

In [17]:
# Embedding dimension: 100
# (For initial testing — this hyperparameter can be tuned later to improve model performance)
glove_file = "glove.twitter.27B.100d.txt"

# Load pre-trained GloVe Twitter embeddings using Gensim
# - 'binary=False' since the GloVe file is in text format
# - 'no_header=True' because GloVe files do not contain a header line
twitter_glove = KeyedVectors.load_word2vec_format(glove_file, binary=False, no_header=True)

# Print the total number of tokens (words) loaded from the GloVe model
print(f"Loaded Twitter GloVe with {len(twitter_glove.key_to_index):,} tokens")

Loaded Twitter GloVe with 1,193,514 tokens


#### Build TensorFlow embedding matrix

In [18]:
# Retrieve the vocabulary learned by the TextVectorization layer
vocab = vectorizer.get_vocabulary()

# Get the dimensionality of the GloVe embeddings (e.g., 100)
embedding_dim = twitter_glove.vector_size

# Initialize an empty embedding matrix
# Each row corresponds to a word in the vocabulary, initialized to zeros
embedding_matrix = np.zeros((len(vocab), embedding_dim))

### OOV handling

#### Random embedding initialization for OOV words

Then we will learn them by training

In [19]:
# Construct the embedding matrix using GloVe vectors or random initialization
# For each word in the vocabulary:
# - If the word exists in the pre-trained GloVe embeddings, use its vector.
# - Otherwise, initialize its vector with random values drawn from a normal distribution
#   (mean = 0, std = 0.6) to represent out-of-vocabulary (OOV) words.
for i, word in enumerate(vocab):
    if word in twitter_glove:
        embedding_matrix[i] = twitter_glove[word]
    else:
        embedding_matrix[i] = np.random.normal(scale=0.6, size=(embedding_dim,))

##### Create Keras Embedding layer

In [20]:
# Create a Keras Embedding layer initialized with the embedding matrix
embedding_layer = layers.Embedding(
    input_dim=len(vocab),                                # Size of the vocabulary
    weights = [embedding_matrix],                        # Weights initialized with embedding matrix
    mask_zero = True,                                    # Ignore padding
    name = 'encoder-embedding',
    output_dim=embedding_dim,                            # Dimension of embedding vectors
    embeddings_initializer=tf.keras.initializers.Constant(embedding_matrix),  # Use precomputed embeddings
    trainable=True                                       # Allow embeddings to be fine-tuned during training
                                                         # (OOV vectors will be updated to more meaningful values)
                                                         # (Pre-trained vectors will also adapt to the specific task)
)

### Model definition

### Bidirectional LSTM Dense Layer

In [21]:
from tensorflow.keras import layers, models

# Baseline model: Bidirectional LSTM + Dense
model_baseline = models.Sequential([
    embedding_layer,                          # Layer defined before
    layers.Bidirectional(layers.LSTM(64)),    # Bidirectional LSTM with 64 units.
    layers.Dense(1, activation='sigmoid')     # Binary output
])

model_baseline.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)


### Baseline Model Training

In [22]:
history = model_baseline.fit(train_ds,
                             validation_data = valid_ds,
                             epochs = 20,
                             verbose = 1)

Epoch 1/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.2945 - loss: 0.7201 - val_accuracy: 0.2938 - val_loss: 0.6039
Epoch 2/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.3508 - loss: 0.5910 - val_accuracy: 0.3333 - val_loss: 0.5677
Epoch 3/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.4818 - loss: 0.2984 - val_accuracy: 0.5254 - val_loss: 0.1382
Epoch 4/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.5954 - loss: -0.4367 - val_accuracy: 0.5367 - val_loss: 0.3544
Epoch 5/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6470 - loss: -1.1810 - val_accuracy: 0.6497 - val_loss: -0.0641
Epoch 6/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.6876 - loss: -1.9154 - val_accuracy: 0.6271 - val_loss: -1.1524
Epoch 7/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6780 - loss: -3.6896 - val_accuracy: 0.5650 - val_loss: -0.8142
Epoch 8/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7343 - loss: -6.3480 - val_accuracy: 0.

In [23]:
test_loss, test_acc = model_baseline.evaluate(test_ds)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")


5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5890 - loss: -3.0473
Test Loss: -4.6467
Test Accuracy: 0.5769
